# 4045. Count Robot Groups — Nhập môn "Mô phỏng sự kiện" → "Quét từ phải sang trái" → Monotonic Stack

Bài toán LeetCode (Medium):

```python
class Solution:
    def countGroups(self, position: list[int], speed: list[int], distance: int) -> int:
        ...
```

## Mục tiêu của notebook

Sau notebook này, bạn sẽ hiểu:

- Cách **mô hình hoá** robot/nhóm robot như các **điểm** chuyển động trên trục số.
- Xử lý **gộp tại thời điểm 0** bằng một lượt quét các khoảng cách liên tiếp.
- Lời giải **mô phỏng theo sự kiện** (event simulation) với phân số chính xác `Fraction` — chậm nhưng rất trực quan, dùng để kiểm chứng.
- Nhận xét then chốt: **nhóm nhanh hơn "đoàn phía trước" thì chắc chắn đuổi kịp**, chậm hơn hoặc bằng thì không bao giờ.
- Lời giải `O(n)` **quét từ phải sang trái** giữ tốc độ nhỏ nhất.
- Lời giải `O(n)` bằng **monotonic stack** quét từ trái sang phải.
- Trace, code chuẩn LeetCode, chứng minh, lỗi hay mắc, bài liên quan (Car Fleet), bài tập có đáp án.

# 1. Đề bài nói gì?

Có `n` robot trên một đường thẳng.

- `position[i]`: vị trí ban đầu của robot `i` tại `t = 0`. Mảng `position` **tăng nghiêm ngặt** (robot `0` ở trái nhất).
- `speed[i]`: tốc độ không đổi của robot `i`, mọi robot đều đi **sang phải**.
- `distance`: ngưỡng gộp.

Quy tắc:

1. Thời gian **liên tục**. Robot/nhóm có tốc độ `v` đi được `v * t` đơn vị sau `t` giây.
2. Khi khoảng cách giữa hai robot/nhóm **`<= distance`**, chúng **gộp** thành một nhóm.
3. Nhiều cặp thoả cùng lúc thì gộp **đồng thời** (cả một chuỗi liên tiếp các nhóm sát nhau cũng gộp thành một).
4. Sau khi gộp, nhóm mới lấy **vị trí và tốc độ của robot bên phải nhất** trong nhóm. Đã gộp thì không bao giờ tách.

Trả về **số nhóm còn lại** sau khi mọi lần gộp có thể đã xảy ra.

### Ví dụ

| position | speed | distance | Kết quả |
|---|---|---:|---:|
| `[1, 5, 6, 20]` | `[4, 3, 2, 3]` | 1 | 2 |
| `[1, 5, 9]` | `[3, 2, 2]` | 2 | 2 |
| `[9]` | `[8]` | 5 | 1 |

Ví dụ 1 diễn ra như sau:

- `t = 0`: R2 (vị trí 5) và R3 (vị trí 6) cách nhau `1 <= 1` → gộp ngay. Nhóm `{R2, R3}` ở vị trí 6, tốc độ 2 (của R3).
- `t = 2`: R1 ở `1 + 4·2 = 9`, nhóm `{R2,R3}` ở `6 + 2·2 = 10` → cách 1 → gộp. Nhóm `{R1,R2,R3}` vị trí 10, tốc độ 2.
- R4 ở `20 + 3t`, chạy nhanh hơn tốc độ 2 → không bao giờ bị đuổi kịp.

Kết quả: 2 nhóm.

### Ràng buộc

- `1 <= n <= 10^5`
- `1 <= position[i], speed[i], distance <= 10^9`

# 2. Mô hình hoá: mỗi nhóm là **một điểm**

Đề nói nhóm lấy **vị trí và tốc độ của robot phải nhất**. Nghĩa là các robot phía sau trong nhóm "không còn quan trọng" — nhóm hoạt động như **một điểm** di chuyển với tốc độ của "đầu tàu".

Vậy ở mọi thời điểm, ta chỉ cần một danh sách các điểm `(vị_trí, tốc_độ)` sắp theo vị trí tăng dần.

Hai nhóm kề nhau `A` (trái) và `B` (phải):

- Nếu `speed_A > speed_B`: khoảng cách **giảm dần** với tốc độ `speed_A - speed_B` → sớm muộn sẽ còn `<= distance` → gộp.
- Nếu `speed_A <= speed_B`: khoảng cách **không giảm** → tự hai nhóm này không bao giờ gộp (trừ khi `B` gộp với ai đó phía trước và **đổi tốc độ**).

Chính chữ "trừ khi" làm bài toán thú vị: tốc độ của nhóm phía trước có thể thay đổi theo thời gian.

# 3. Bước 1 — Gộp tại thời điểm `t = 0`

Ở `t = 0`, mọi cặp robot liền kề có `position[i] - position[i-1] <= distance` đều gộp ngay (gộp theo chuỗi).

Quét từ trái sang phải, giữ danh sách nhóm. Với robot `i`:

- Nếu nó **sát** robot `i - 1` (khoảng cách `<= distance`): nó nhập vào nhóm cuối, và trở thành **đầu tàu mới** (vì nó ở bên phải) → **thay** vị trí/tốc độ của nhóm cuối bằng của nó.
- Ngược lại: mở nhóm mới.

Chú ý: so với robot `i - 1` (robot liền trước), **không** so với vị trí của nhóm. Thật ra hai cách giống nhau, vì robot `i - 1` luôn là đầu tàu của nhóm cuối.

In [1]:
def merge_at_time_zero(position, speed, distance):
    groups = []                                  # mỗi phần tử: [vị_trí, tốc_độ, số_robot]
    for i in range(len(position)):
        if groups and position[i] - position[i - 1] <= distance:
            groups[-1][0] = position[i]          # robot i là đầu tàu mới
            groups[-1][1] = speed[i]
            groups[-1][2] += 1
        else:
            groups.append([position[i], speed[i], 1])
    return groups


print(merge_at_time_zero([1, 5, 6, 20], [4, 3, 2, 3], 1))   # R2, R3 gộp
print(merge_at_time_zero([1, 5, 9], [3, 2, 2], 2))          # chưa ai gộp
print(merge_at_time_zero([1, 2, 3, 10], [5, 1, 7, 2], 1))   # R1, R2, R3 gộp theo chuỗi

[[1, 4, 1], [6, 2, 2], [20, 3, 1]]
[[1, 3, 1], [5, 2, 1], [9, 2, 1]]
[[3, 7, 3], [10, 2, 1]]


Ví dụ cuối: `1, 2, 3` cách nhau từng bước 1 → **cả chuỗi** gộp thành một nhóm, đầu tàu là robot ở vị trí 3 (tốc độ 7). Tốc độ 5 và 1 của hai robot sau **bị bỏ qua**.

# 4. Cách 1 — Mô phỏng theo sự kiện (chính xác tuyệt đối)

Sau bước 1, mọi nhóm kề nhau đều cách nhau `> distance`. Ta mô phỏng:

1. Với mỗi cặp kề nhau `(A, B)` có `speed_A > speed_B`: thời điểm gộp là
   ```text
   t = (gap - distance) / (speed_A - speed_B)      với gap = pos_B - pos_A
   ```
   (thời gian để khoảng cách giảm từ `gap` xuống đúng `distance`).
2. Lấy `t` **nhỏ nhất** — sự kiện gộp sớm nhất. Nếu không có cặp nào → dừng.
3. Cho mọi nhóm chạy thêm `t` giây.
4. Gộp mọi cặp kề nhau có khoảng cách `<= distance` (giống bước 1).
5. Lặp lại.

## Vì sao cần `Fraction`?

`t` là phân số, ví dụ `1/3`. Số thực (`float`) không biểu diễn chính xác `1/3`, sau vài phép tính có thể ra khoảng cách `1.0000000000000002` thay vì `1` → **lỡ** một lần gộp. `fractions.Fraction` lưu tử/mẫu số nguyên → **chính xác tuyệt đối**.

In [2]:
from fractions import Fraction

print(0.1 + 0.2 == 0.3)                                       # False! lỗi làm tròn
print(Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10))   # True
t = Fraction(1, 3)
print("t =", t, "| 3 * t =", 3 * t, "| 5 + 2*t =", 5 + 2 * t)

False
True
t = 1/3 | 3 * t = 1 | 5 + 2*t = 17/3


In [3]:
def count_groups_simulation(position, speed, distance, verbose=False):
    groups = [[Fraction(p), v, c] for p, v, c in merge_at_time_zero(position, speed, distance)]
    now = Fraction(0)
    if verbose:
        print(f"t = 0: {[(str(p), v) for p, v, c in groups]}")

    while True:
        # 1) tìm sự kiện gộp sớm nhất
        best_t = None
        for A, B in zip(groups, groups[1:]):
            if A[1] > B[1]:
                t = (B[0] - A[0] - distance) / Fraction(A[1] - B[1])
                if best_t is None or t < best_t:
                    best_t = t
        if best_t is None:
            break
        # 2) mọi nhóm chạy thêm best_t giây
        now += best_t
        for g in groups:
            g[0] += g[1] * best_t
        # 3) gộp các nhóm kề nhau đủ gần (gộp theo chuỗi, lấy đầu tàu bên phải)
        merged = [groups[0]]
        for g in groups[1:]:
            if g[0] - merged[-1][0] <= distance:
                g[2] += merged[-1][2]
                merged[-1] = g
            else:
                merged.append(g)
        groups = merged
        if verbose:
            print(f"t = {now}: {[(str(p), v) for p, v, c in groups]}")
    return len(groups)


print(count_groups_simulation([1, 5, 6, 20], [4, 3, 2, 3], 1, verbose=True))
print()
print(count_groups_simulation([1, 5, 9], [3, 2, 2], 2, verbose=True))

t = 0: [('1', 4), ('6', 2), ('20', 3)]
t = 2: [('10', 2), ('26', 3)]
2

t = 0: [('1', 3), ('5', 2), ('9', 2)]
t = 2: [('9', 2), ('13', 2)]
2


Mô phỏng ra đúng kết quả của đề. Nhưng mỗi vòng lặp duyệt mọi cặp (`O(n)`) và mỗi vòng chỉ chắc chắn gộp được **ít nhất một** cặp → tối đa `n` vòng → `O(n²)`. Với `n = 10^5`, quá chậm.

Mô phỏng vẫn rất quý: nó là **"trọng tài"** để kiểm chứng lời giải nhanh mà ta sắp xây dựng.

# 5. Nhận xét then chốt — nghĩ từ **phải sang trái**

Nhóm **phải nhất** không có ai phía trước → **không bao giờ đổi tốc độ**. Gọi tốc độ đó là `cur`.

Giờ xét nhóm ngay bên trái nó, tốc độ `v`:

- `v > cur`: nhóm này nhanh hơn → khoảng cách giảm → **chắc chắn gộp** vào đoàn phía trước. Sau khi gộp, đoàn vẫn có tốc độ `cur` (tốc độ của đầu tàu bên phải).
- `v <= cur`: không bao giờ đuổi kịp → nó là **một nhóm cuối cùng riêng biệt**, và trở thành "đoàn phía trước" mới cho các nhóm bên trái, với `cur = v`.

Tiếp tục sang trái. Tổng quát:

> Quét các nhóm (sau bước `t = 0`) **từ phải sang trái**, giữ `cur` = tốc độ của **đoàn gần nhất phía trước**.
> Nhóm có `v > cur` → bị hút vào đoàn đó. Nhóm có `v <= cur` → thành đoàn mới, `cur = v`.
> Đáp án = số đoàn được tạo.

Ví dụ 1 sau bước 1: nhóm (vị trí, tốc độ) = `(1, 4)`, `(6, 2)`, `(20, 3)`.

| Nhóm (từ phải) | v | cur trước | v > cur? | Kết luận | cur sau | số đoàn |
|---|---:|---:|---|---|---:|---:|
| (20, 3) | 3 | ∞ | không | đoàn mới | 3 | 1 |
| (6, 2) | 2 | 3 | không | đoàn mới | 2 | 2 |
| (1, 4) | 4 | 2 | **có** | gộp vào đoàn (6, 2) | 2 | 2 |

Đáp án `2` ✔.

## Vì sao `cur` là "tốc độ nhỏ nhất bên phải"?

Mỗi khi tạo đoàn mới, `v <= cur` → `cur` chỉ có thể **giảm hoặc giữ nguyên**. Vậy `cur` luôn là **tốc độ nhỏ nhất** trong các nhóm đã xét (bên phải). Lời giải tương đương:

> Một nhóm là "đoàn cuối cùng" ⇔ tốc độ của nó **`<=` mọi tốc độ của các nhóm bên phải nó**.

Đáp án = số nhóm thoả điều kiện này (gọi là **cực tiểu hậu tố** — suffix minimum).

## Vì sao `v == cur` thì không gộp?

Sau bước `t = 0`, mọi nhóm kề nhau cách nhau `> distance`. Hai nhóm cùng tốc độ giữ nguyên khoảng cách mãi mãi → không bao giờ `<= distance`.

In [4]:
def count_groups_right_to_left(position, speed, distance):
    groups = merge_at_time_zero(position, speed, distance)
    fleets = 0
    cur = float("inf")              # chưa có đoàn nào phía trước
    for pos, v, cnt in reversed(groups):
        if v <= cur:                # không đuổi kịp đoàn phía trước -> đoàn mới
            fleets += 1
            cur = v
        # ngược lại: v > cur -> bị hút vào đoàn phía trước, cur giữ nguyên
    return fleets


print(count_groups_right_to_left([1, 5, 6, 20], [4, 3, 2, 3], 1))   # 2
print(count_groups_right_to_left([1, 5, 9], [3, 2, 2], 2))          # 2
print(count_groups_right_to_left([9], [8], 5))                      # 1

2
2
1


`reversed(groups)` duyệt list từ cuối về đầu mà không tạo bản sao.

## Trace ví dụ 2

`position = [1, 5, 9]`, `speed = [3, 2, 2]`, `distance = 2`. Bước 1: không ai gộp (khoảng cách 4, 4).

| Nhóm (từ phải) | v | cur trước | Kết luận | cur sau | số đoàn |
|---|---:|---:|---|---:|---:|
| (9, 2) | 2 | ∞ | đoàn mới | 2 | 1 |
| (5, 2) | 2 | 2 | `2 <= 2` → đoàn mới (cùng tốc độ, không đuổi kịp) | 2 | 2 |
| (1, 3) | 3 | 2 | `3 > 2` → gộp vào đoàn (5, 2) | 2 | 2 |

Đáp án `2` ✔.

In [5]:
def trace_right_to_left(position, speed, distance):
    groups = merge_at_time_zero(position, speed, distance)
    print("Sau gộp t=0 (vị trí, tốc độ, số robot):", groups)
    fleets, cur = 0, float("inf")
    for pos, v, cnt in reversed(groups):
        if v <= cur:
            fleets += 1
            print(f"nhóm ở {pos} (v={v}): v <= cur={cur} -> ĐOÀN MỚI, cur = {v}, số đoàn = {fleets}")
            cur = v
        else:
            print(f"nhóm ở {pos} (v={v}): v > cur={cur} -> đuổi kịp, gộp vào đoàn phía trước")
    print("Đáp án:", fleets)
    return fleets

trace_right_to_left([1, 5, 6, 20], [4, 3, 2, 3], 1)
print()
trace_right_to_left([1, 5, 9], [3, 2, 2], 2)

Sau gộp t=0 (vị trí, tốc độ, số robot): [[1, 4, 1], [6, 2, 2], [20, 3, 1]]
nhóm ở 20 (v=3): v <= cur=inf -> ĐOÀN MỚI, cur = 3, số đoàn = 1
nhóm ở 6 (v=2): v <= cur=3 -> ĐOÀN MỚI, cur = 2, số đoàn = 2
nhóm ở 1 (v=4): v > cur=2 -> đuổi kịp, gộp vào đoàn phía trước
Đáp án: 2

Sau gộp t=0 (vị trí, tốc độ, số robot): [[1, 3, 1], [5, 2, 1], [9, 2, 1]]
nhóm ở 9 (v=2): v <= cur=inf -> ĐOÀN MỚI, cur = 2, số đoàn = 1
nhóm ở 5 (v=2): v <= cur=2 -> ĐOÀN MỚI, cur = 2, số đoàn = 2
nhóm ở 1 (v=3): v > cur=2 -> đuổi kịp, gộp vào đoàn phía trước
Đáp án: 2


2

# 6. Cách 3 — Monotonic Stack quét từ trái sang phải

Ta cũng có thể quét **trái sang phải** với một stack chứa tốc độ các "đoàn tạm thời":

Khi nhóm mới tốc độ `v` xuất hiện (ở bên phải mọi nhóm trong stack):

- Mọi đoàn trong stack có tốc độ **`> v`** sẽ đuổi kịp (nhóm mới hoặc đoàn mà nhóm mới thuộc về) → **pop** chúng (chúng không còn là đoàn cuối cùng).
- Push `v`.

Cuối cùng, stack chứa đúng các nhóm có tốc độ `<=` mọi tốc độ bên phải — chính là các đoàn cuối cùng. Stack luôn **không giảm** từ đáy lên đỉnh.

Đây đúng là tag "Monotonic Stack" của bài.

In [6]:
def count_groups_stack(position, speed, distance):
    groups = merge_at_time_zero(position, speed, distance)
    stack = []                       # tốc độ, không giảm từ đáy lên đỉnh
    for pos, v, cnt in groups:
        while stack and stack[-1] > v:
            stack.pop()              # đoàn nhanh hơn phía sau sẽ đuổi kịp
        stack.append(v)
    return len(stack)


print(count_groups_stack([1, 5, 6, 20], [4, 3, 2, 3], 1))   # 2
print(count_groups_stack([1, 5, 9], [3, 2, 2], 2))          # 2
print(count_groups_stack([9], [8], 5))                      # 1

2
2
1


In [7]:
def trace_stack(position, speed, distance):
    groups = merge_at_time_zero(position, speed, distance)
    stack = []
    for pos, v, cnt in groups:
        popped = []
        while stack and stack[-1] > v:
            popped.append(stack.pop())
        stack.append(v)
        print(f"nhóm ở {pos} (v={v}): pop {popped if popped else '-'} -> stack = {stack}")
    print("Đáp án = len(stack) =", len(stack))

trace_stack([1, 5, 6, 20], [4, 3, 2, 3], 1)
print()
trace_stack([3, 10, 20, 30, 40], [5, 9, 4, 6, 1], 2)

nhóm ở 1 (v=4): pop - -> stack = [4]
nhóm ở 6 (v=2): pop [4] -> stack = [2]
nhóm ở 20 (v=3): pop - -> stack = [2, 3]
Đáp án = len(stack) = 2

nhóm ở 3 (v=5): pop - -> stack = [5]
nhóm ở 10 (v=9): pop - -> stack = [5, 9]
nhóm ở 20 (v=4): pop [9, 5] -> stack = [4]
nhóm ở 30 (v=6): pop - -> stack = [4, 6]
nhóm ở 40 (v=1): pop [6, 4] -> stack = [1]
Đáp án = len(stack) = 1


# 7. Lời giải LeetCode chuẩn

Gộp bước `t = 0` và quét phải → trái vào một hàm, không cần lưu list nhóm:

- Duyệt `i` từ `n - 1` về `0`.
- Robot `i` là **đầu tàu của nhóm `t = 0`** của nó ⇔ `i == n - 1` hoặc `position[i + 1] - position[i] > distance` (robot bên phải nó **không** sát nó).
- Chỉ đầu tàu mới tham gia so sánh với `cur`; các robot khác bị "nuốt" ngay ở `t = 0`.

In [8]:
class Solution:
    def countGroups(self, position: list[int], speed: list[int], distance: int) -> int:
        n = len(position)
        fleets = 0
        cur = float("inf")
        for i in range(n - 1, -1, -1):
            is_leader = (i == n - 1) or (position[i + 1] - position[i] > distance)
            if not is_leader:
                continue                     # đã gộp vào nhóm bên phải ngay tại t = 0
            if speed[i] <= cur:
                fleets += 1
                cur = speed[i]
        return fleets

`range(n - 1, -1, -1)` chạy `i = n-1, n-2, ..., 0` (bước `-1`, dừng **trước** `-1`).

In [9]:
sol = Solution()

tests = [
    ([1, 5, 6, 20], [4, 3, 2, 3], 1, 2),
    ([1, 5, 9], [3, 2, 2], 2, 2),
    ([9], [8], 5, 1),
    ([1, 2, 3], [1, 1, 1], 1, 1),               # gộp hết ngay t = 0
    ([1, 10, 20], [1, 2, 3], 1, 3),              # càng phía trước càng nhanh: không ai gộp
    ([1, 10, 20], [3, 2, 1], 1, 1),              # càng phía trước càng chậm: gộp hết
    ([1, 2, 10], [9, 1, 5], 1, 2),               # bẫy: xem giải thích bên dưới
    ([1, 3, 100], [100, 1, 2], 1, 2),
]
for position, speed, distance, expected in tests:
    result = sol.countGroups(position, speed, distance)
    print(f"{str(position):14} {str(speed):14} d={distance} -> {result} | expected {expected} | "
          f"{'PASS' if result == expected else 'FAIL'}")
    assert result == expected

[1, 5, 6, 20]  [4, 3, 2, 3]   d=1 -> 2 | expected 2 | PASS
[1, 5, 9]      [3, 2, 2]      d=2 -> 2 | expected 2 | PASS
[9]            [8]            d=5 -> 1 | expected 1 | PASS
[1, 2, 3]      [1, 1, 1]      d=1 -> 1 | expected 1 | PASS
[1, 10, 20]    [1, 2, 3]      d=1 -> 3 | expected 3 | PASS
[1, 10, 20]    [3, 2, 1]      d=1 -> 1 | expected 1 | PASS
[1, 2, 10]     [9, 1, 5]      d=1 -> 2 | expected 2 | PASS
[1, 3, 100]    [100, 1, 2]    d=1 -> 2 | expected 2 | PASS


### Test "bẫy": `([1, 2, 10], [9, 1, 5], 1)`

Nhìn vội, R1 có tốc độ 9 — nhanh nhất — nên dễ nghĩ "R1 sẽ đuổi kịp tất cả, đáp án 1". **Sai!**

- Ở `t = 0`, R1 và R2 cách nhau 1 → gộp ngay. Nhóm `{R1, R2}` lấy tốc độ của **R2** (robot bên phải) là **1**. Tốc độ 9 của R1 "biến mất".
- R3 ở phía trước với tốc độ 5 > 1 → nhóm `{R1, R2}` không bao giờ đuổi kịp.

Đáp án là **2**. Mô phỏng chính xác xác nhận điều này:

In [10]:
print("Mô phỏng chính xác:", count_groups_simulation([1, 2, 10], [9, 1, 5], 1, verbose=True))

t = 0: [('2', 1), ('10', 5)]
Mô phỏng chính xác: 2


In [11]:
import random, time

random.seed(45)
for _ in range(3000):
    n = random.randint(1, 8)
    position = sorted(random.sample(range(1, 40), n))
    speed = [random.randint(1, 6) for _ in range(n)]
    distance = random.randint(1, 6)
    expected = count_groups_simulation(position, speed, distance)
    assert count_groups_right_to_left(position, speed, distance) == expected
    assert count_groups_stack(position, speed, distance) == expected
    assert sol.countGroups(position, speed, distance) == expected
print("Cả 3 cách nhanh khớp với mô phỏng chính xác trên 3000 bộ test ✔")

n = 10**5
position = sorted(random.sample(range(1, 10**9), n))
speed = [random.randint(1, 10**9) for _ in range(n)]
t0 = time.time()
sol.countGroups(position, speed, random.randint(1, 10**4))
print(f"n = 1e5 chạy trong {time.time() - t0:.3f} giây")

Cả 3 cách nhanh khớp với mô phỏng chính xác trên 3000 bộ test ✔
n = 1e5 chạy trong 0.014 giây


# 8. Vì sao thuật toán đúng? (chứng minh)

Sau bước `t = 0`, gọi các nhóm là `G1, G2, ..., Gm` (trái → phải), tốc độ `v1, ..., vm`, khoảng cách giữa các nhóm kề nhau `> distance`.

**Khẳng định:** `Gk` không bao giờ bị gộp vào nhóm bên phải ⇔ `vk <= min(v(k+1), ..., vm)`.

**(⇐) Nếu `vk <= min` bên phải:** Gọi `N(t)` là nhóm ngay bên phải `Gk` tại thời điểm `t`. Vị trí của `N(t)` là vị trí của một robot đầu tàu nào đó bên phải, chạy với tốc độ `>= min >= vk`. Khi `N` gộp với nhóm phía trước, vị trí của nó **nhảy về phía trước** (lấy vị trí đầu tàu mới). Vậy khoảng cách từ `Gk` tới nhóm kề phải **không bao giờ giảm**, mà ban đầu `> distance` → không bao giờ gộp.

**(⇒) Nếu `vk > min` bên phải:** Gọi `Gj` là nhóm có tốc độ nhỏ nhất bên phải (lấy `j` lớn nhất nếu nhiều). Áp dụng (⇐) cho `Gj`: nó không bao giờ gộp với ai phía trước nên luôn chạy tốc độ `vj < vk`. Mọi nhóm giữa `Gk` và `Gj` nằm giữa hai điểm đó. Khoảng cách `Gk → Gj` giảm đều với tốc độ `>= vk - vj > 0`, nên trước khi `Gk` "vượt" tới `Gj`, khoảng cách tới nhóm kề phải chắc chắn đạt `<= distance` → `Gk` gộp.

**Kết luận:** số nhóm cuối cùng = số `k` thoả `vk <= min(v(k+1..m))` (với nhóm cuối luôn thoả). Quét phải → trái với `cur = min` chính là đếm các `k` này. Stack trái → phải giữ đúng tập này.

# 9. Độ phức tạp

Gọi `n` là số robot.

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| 1. Mô phỏng sự kiện (Fraction) | `O(n²)` (và phép tính phân số chậm) | `O(n)` |
| 2. Quét phải → trái | `O(n)` | `O(1)` (bản LeetCode) |
| 3. Monotonic stack | `O(n)` (mỗi nhóm push/pop tối đa 1 lần) | `O(n)` |

# 10. Những lỗi thường gặp

## Lỗi 1 — Bỏ qua bước gộp tại `t = 0`

Nếu bỏ qua, ví dụ 1 sẽ xét R2 (tốc độ 3) như một nhóm riêng, trong khi nó đã gộp với R3 (tốc độ 2) ngay từ đầu.

## Lỗi 2 — Dùng tốc độ sai cho nhóm gộp tại `t = 0`

Nhóm lấy tốc độ của robot **phải nhất**, không phải trái nhất, không phải nhỏ nhất/lớn nhất.

## Lỗi 3 — Coi tốc độ bằng nhau là đuổi kịp

Dùng `v < cur` thay vì `v <= cur` cho điều kiện "đoàn mới" → hai nhóm cùng tốc độ bị đếm nhầm là gộp.

## Lỗi 4 — Chỉ so với nhóm **kề** bên phải

So `v` với tốc độ nhóm kề (thay vì `min` bên phải) → sai khi nhóm kề nhanh nhưng phía trước nữa có đoàn chậm.

## Lỗi 5 — Dùng `float` trong mô phỏng

Lỗi làm tròn có thể bỏ sót đúng lúc khoảng cách bằng `distance`.

In [12]:
def wrong_skip_t0(position, speed, distance):
    fleets, cur = 0, float("inf")
    for v in reversed(speed):            # bỏ qua gộp t = 0
        if v <= cur:
            fleets += 1
            cur = v
    return fleets

def wrong_compare_neighbor(position, speed, distance):
    groups = merge_at_time_zero(position, speed, distance)
    fleets = 1
    for k in range(len(groups) - 1):
        if groups[k][1] <= groups[k + 1][1]:   # chỉ so với nhóm kề
            fleets += 1
    return fleets

ex = ([1, 2, 10], [1, 9, 5], 1)        # R1, R2 gộp ngay t = 0 -> nhóm tốc độ 9 đuổi kịp R3
print("Lỗi 1 | sai:", wrong_skip_t0(*ex), "| đúng:", sol.countGroups(*ex), "| mô phỏng:", count_groups_simulation(*ex))
ex2 = ([1, 10, 20], [2, 5, 1], 1)
print("Lỗi 4 | sai:", wrong_compare_neighbor(*ex2), "| đúng:", sol.countGroups(*ex2),
      "| mô phỏng:", count_groups_simulation(*ex2))

Lỗi 1 | sai: 2 | đúng: 1 | mô phỏng: 1
Lỗi 4 | sai: 2 | đúng: 1 | mô phỏng: 1


`f(*ex)`: dấu `*` **mở gói** tuple `ex` thành các tham số riêng: `f(*([1,5,6,20], [4,3,2,3], 1))` ≡ `f([1,5,6,20], [4,3,2,3], 1)`.

Lỗi 1 với `([1, 2, 10], [1, 9, 5])`: nếu bỏ qua bước `t = 0`, R1 (tốc độ 1) bị coi là một nhóm riêng chậm chạp. Thực tế R1 đã gộp với R2 ngay từ đầu và nhóm chạy tốc độ 9 → đuổi kịp R3.

Lỗi 4 với `([1, 10, 20], [2, 5, 1])`: nhóm ở 1 (tốc độ 2) chậm hơn nhóm kề (5) nên "tưởng" là đoàn riêng. Nhưng nhóm ở 10 sẽ bị hút vào nhóm ở 20 (tốc độ 1), và đoàn đó chạy tốc độ 1 < 2 → nhóm ở 1 cũng đuổi kịp.

# 11. Mẹo nhận diện pattern

- Các vật **cùng chiều**, vật sau đuổi vật trước, **gộp thì lấy tốc độ vật trước** → nghĩ **từ phía trước ra phía sau** (phải → trái): vật trước quyết định số phận vật sau.
- Câu hỏi "đếm số nhóm cuối cùng" → đếm **cực tiểu hậu tố** / dùng **monotonic stack**.
- Mô phỏng thời gian liên tục → dùng **phân số chính xác** khi kiểm chứng.
- Bài cùng họ: **Car Fleet** (LeetCode 853), **Car Fleet II** (LeetCode 1776).

# 12. Bài liên quan — Car Fleet

Đề Car Fleet: `n` xe chạy về đích `target`, xe `i` ở `position[i]` tốc độ `speed[i]`. Xe không được vượt; đuổi kịp xe trước thì đi cùng (thành một đoàn, tốc độ của xe trước). Đếm số đoàn **về tới đích**.

Khác biệt: có **đích** — xe sau chỉ thành cùng đoàn nếu đuổi kịp **trước khi tới đích**. Mẹo: tính **thời gian tới đích** `time = (target - pos) / speed`, quét từ xe gần đích nhất: xe có `time` **lớn hơn** đoàn phía trước thì thành đoàn mới.

In [13]:
def car_fleet(target, position, speed):
    cars = sorted(zip(position, speed), reverse=True)    # gần đích trước
    fleets, slowest_time = 0, 0.0
    for pos, v in cars:
        time_to_target = (target - pos) / v
        if time_to_target > slowest_time:                # không đuổi kịp đoàn phía trước
            fleets += 1
            slowest_time = time_to_target
    return fleets

print(car_fleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]))   # 3 (ví dụ kinh điển của LeetCode 853)

3


# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

`position = [2, 4, 9, 15, 16, 30]`, `speed = [7, 1, 6, 5, 3, 4]`, `distance = 2`.

1. Gộp tại `t = 0`: nhóm nào hình thành, đầu tàu là ai?
2. Quét phải → trái với `cur`: đáp án bao nhiêu?

## Bài 2 — Dự đoán output

```python
merge_at_time_zero([1, 3, 5, 8], [9, 8, 7, 6], 2)
list(range(4, -1, -1))
```

In [14]:
print(merge_at_time_zero([1, 3, 5, 8], [9, 8, 7, 6], 2))
print(list(range(4, -1, -1)))

[[5, 7, 3], [8, 6, 1]]
[4, 3, 2, 1, 0]


## Bài 3 — Điền chỗ trống: gộp tại `t = 0`

Trả về **list tốc độ** của các nhóm sau khi gộp tại `t = 0`.

In [15]:
def exercise_3_speeds_after_t0(position, speed, distance):
    speeds = []
    # TODO: với mỗi i:
    #   nếu speeds không rỗng và position[i] - position[i-1] <= distance: speeds[-1] = speed[i]
    #   ngược lại: speeds.append(speed[i])
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")

# assert exercise_3_speeds_after_t0([1, 5, 6, 20], [4, 3, 2, 3], 1) == [4, 2, 3]

## Bài 4 — Tự viết lời giải phải → trái (không cần list nhóm)

In [16]:
def exercise_4_count_groups(position, speed, distance):
    # TODO
    raise NotImplementedError

## Bài 5 — Car Fleet bằng số nguyên

Viết lại `car_fleet` mà **không dùng phép chia** (tránh `float`): so sánh `(target - p1) / v1 > (target - p2) / v2` bằng cách **nhân chéo**.

In [17]:
def exercise_5_car_fleet_integer(target, position, speed):
    # TODO
    raise NotImplementedError

## Bài 6 — Mini challenge: kích thước từng nhóm cuối

Trả về list **số robot** trong mỗi nhóm cuối cùng, theo thứ tự trái → phải.

Ví dụ 1: `[3, 1]` (nhóm `{R1,R2,R3}` và `{R4}`).

In [18]:
def exercise_6_group_sizes(position, speed, distance):
    # TODO: monotonic stack lưu [tốc_độ, số_robot]; khi pop, cộng số robot vào nhóm mới
    raise NotImplementedError

## Gợi ý theo mức độ

**Bài 4**
- Mức 1: robot `i` là đầu tàu nhóm `t = 0` ⇔ `i == n-1` hoặc `position[i+1] - position[i] > distance`.
- Mức 2: duyệt `i` từ `n-1` về `0`, bỏ qua robot không phải đầu tàu.
- Mức 3: `if speed[i] <= cur: fleets += 1; cur = speed[i]`.

**Bài 5**
- Mức 1: `a / b > c / d` với `b, d > 0` ⇔ `a * d > c * b`.
- Mức 2: lưu "đoàn chậm nhất phía trước" dưới dạng cặp `(quãng_đường, tốc_độ)`.

**Bài 6**
- Mức 1: một nhóm bị pop sẽ gộp vào đoàn của nhóm vừa đến.
- Mức 2: `cnt = số robot của nhóm mới`; `while stack and stack[-1][0] > v: cnt += stack.pop()[1]`.
- Mức 3: `stack.append([v, cnt])`; đáp án là `[c for v, c in stack]`.

# 14. Đáp án tham khảo

**Bài 1:**

- `t = 0`: khoảng cách liên tiếp `2, 5, 6, 1, 14`. Cặp `(2,4)` cách 2 → gộp, đầu tàu vị trí 4 (tốc độ 1). Cặp `(15,16)` cách 1 → gộp, đầu tàu vị trí 16 (tốc độ 3).
- Các nhóm: `(4, v=1)`, `(9, v=6)`, `(16, v=3)`, `(30, v=4)`.
- Quét phải → trái: `30 (v=4)`: đoàn mới, cur=4 → `16 (v=3)`: 3 ≤ 4 đoàn mới, cur=3 → `9 (v=6)`: 6 > 3 gộp → `4 (v=1)`: 1 ≤ 3 đoàn mới, cur=1.
- Đáp án **3**.

**Bài 2:** `[[5, 7, 3], [8, 6, 1]]` — robot ở `1, 3, 5` gộp theo chuỗi (khoảng cách 2, 2), đầu tàu ở 5 tốc độ 7; robot ở 8 cách 3 `> 2` nên đứng riêng. Và `[4, 3, 2, 1, 0]`.

In [19]:
ex1 = ([2, 4, 9, 15, 16, 30], [7, 1, 6, 5, 3, 4], 2)
assert sol.countGroups(*ex1) == 3 == count_groups_simulation(*ex1)
print("Đáp án bài 1 đã được kiểm chứng bằng mô phỏng ✔")

Đáp án bài 1 đã được kiểm chứng bằng mô phỏng ✔


In [20]:
def answer_3_speeds_after_t0(position, speed, distance):
    speeds = []
    for i in range(len(position)):
        if speeds and position[i] - position[i - 1] <= distance:
            speeds[-1] = speed[i]
        else:
            speeds.append(speed[i])
    return speeds

assert answer_3_speeds_after_t0([1, 5, 6, 20], [4, 3, 2, 3], 1) == [4, 2, 3]
print("Đáp án bài 3 PASS")

Đáp án bài 3 PASS


In [21]:
def answer_4_count_groups(position, speed, distance):
    n = len(position)
    fleets, cur = 0, float("inf")
    for i in range(n - 1, -1, -1):
        if i < n - 1 and position[i + 1] - position[i] <= distance:
            continue
        if speed[i] <= cur:
            fleets += 1
            cur = speed[i]
    return fleets

random.seed(4)
for _ in range(1000):
    n = random.randint(1, 8)
    position = sorted(random.sample(range(1, 40), n))
    speed = [random.randint(1, 6) for _ in range(n)]
    d = random.randint(1, 6)
    assert answer_4_count_groups(position, speed, d) == count_groups_simulation(position, speed, d)
print("Đáp án bài 4 PASS")

Đáp án bài 4 PASS


In [22]:
def answer_5_car_fleet_integer(target, position, speed):
    cars = sorted(zip(position, speed), reverse=True)
    fleets = 0
    lead_dist, lead_speed = 0, 1          # "thời gian" 0/1 = 0
    for pos, v in cars:
        dist = target - pos
        # dist / v > lead_dist / lead_speed  <=>  dist * lead_speed > lead_dist * v
        if dist * lead_speed > lead_dist * v:
            fleets += 1
            lead_dist, lead_speed = dist, v
    return fleets

assert answer_5_car_fleet_integer(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]) == 3
random.seed(5)
for _ in range(1000):
    target = random.randint(10, 30)
    n = random.randint(1, 6)
    position = random.sample(range(0, target), n)
    speed = [random.randint(1, 5) for _ in range(n)]
    assert answer_5_car_fleet_integer(target, position, speed) == car_fleet(target, position, speed)
print("Đáp án bài 5 PASS")

Đáp án bài 5 PASS


In [23]:
def answer_6_group_sizes(position, speed, distance):
    stack = []                                  # [tốc_độ, số_robot]
    for pos, v, cnt in merge_at_time_zero(position, speed, distance):
        while stack and stack[-1][0] > v:
            cnt += stack.pop()[1]
        stack.append([v, cnt])
    return [c for v, c in stack]

def group_sizes_simulation(position, speed, distance):
    groups = [[Fraction(p), v, c] for p, v, c in merge_at_time_zero(position, speed, distance)]
    while True:
        best_t = None
        for A, B in zip(groups, groups[1:]):
            if A[1] > B[1]:
                t = (B[0] - A[0] - distance) / Fraction(A[1] - B[1])
                best_t = t if best_t is None else min(best_t, t)
        if best_t is None:
            return [c for p, v, c in groups]
        for g in groups:
            g[0] += g[1] * best_t
        merged = [groups[0]]
        for g in groups[1:]:
            if g[0] - merged[-1][0] <= distance:
                g[2] += merged[-1][2]
                merged[-1] = g
            else:
                merged.append(g)
        groups = merged

assert answer_6_group_sizes([1, 5, 6, 20], [4, 3, 2, 3], 1) == [3, 1]
random.seed(6)
for _ in range(1000):
    n = random.randint(1, 8)
    position = sorted(random.sample(range(1, 40), n))
    speed = [random.randint(1, 6) for _ in range(n)]
    d = random.randint(1, 6)
    assert answer_6_group_sizes(position, speed, d) == group_sizes_simulation(position, speed, d)
print("Đáp án bài 6 PASS")

Đáp án bài 6 PASS


# 15. Tổng kết

## Kiến thức Python

```python
from fractions import Fraction      # số hữu tỉ chính xác
reversed(lst), range(n - 1, -1, -1) # duyệt ngược
zip(groups, groups[1:])             # cặp kề nhau
stack.append / stack.pop / stack[-1]
f(*args)                            # mở gói tham số
float("inf")
```

## Kỹ thuật thuật toán

1. **Mô hình hoá** nhóm = một điểm (đầu tàu).
2. **Gộp tại t = 0** bằng một lượt quét khoảng cách liên tiếp.
3. **Mô phỏng sự kiện** với `Fraction` để kiểm chứng.
4. **Quét phải → trái**, giữ `cur = min tốc độ phía trước`: `v <= cur` → đoàn mới.
5. Tương đương: **monotonic stack** trái → phải, pop tốc độ `> v`.

## Công thức nhớ nhanh

> Đáp án = số đầu tàu (sau gộp `t = 0`) có tốc độ **`<=` mọi tốc độ đầu tàu bên phải**.

## Độ phức tạp

```text
Time:  O(n)
Space: O(1)  (bản quét phải -> trái)
```

## Checklist trước khi nộp

- [ ] Đã gộp các robot sát nhau tại `t = 0` (theo chuỗi)?
- [ ] Tốc độ nhóm là của robot **phải nhất**?
- [ ] Điều kiện đoàn mới là `v <= cur` (bằng nhau không gộp)?
- [ ] So với **min** bên phải, không chỉ nhóm kề?

# 16. Phiên bản cuối cùng nên nhớ

```python
class Solution:
    def countGroups(self, position: list[int], speed: list[int], distance: int) -> int:
        n = len(position)
        fleets = 0
        cur = float("inf")
        for i in range(n - 1, -1, -1):
            if i < n - 1 and position[i + 1] - position[i] <= distance:
                continue                 # gộp vào nhóm bên phải ngay tại t = 0
            if speed[i] <= cur:
                fleets += 1
                cur = speed[i]
        return fleets
```

Ý tưởng cốt lõi: **người phía trước quyết định số phận người phía sau** — nên hãy nhìn từ phía trước (phải) về phía sau (trái).